# SecureSight Word Report Generator

This notebook generates two professional Microsoft Word reports from the `evaluation` directory:

1. **Full Evaluation Report** including all evaluation results and statistical analysis.
2. **Statistical Analysis Only Report** focused strictly on the `statistical_analysis` outputs.


In [2]:
import os
import json
from pathlib import Path

EVAL_DIR = Path('evaluation')
STAT_ANALYSIS_DIR = EVAL_DIR / 'statistical_analysis'
OUTPUT_DIR = STAT_ANALYSIS_DIR / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Evaluation directory: {EVAL_DIR.resolve()}')
print(f'Statistical analysis directory: {STAT_ANALYSIS_DIR.resolve()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')

subdirs = [p.name for p in EVAL_DIR.iterdir() if p.is_dir()]
print('Found evaluation subdirectories:', subdirs)


Evaluation directory: E:\tasnim\Graduation project model\tasnim\evaluation
Statistical analysis directory: E:\tasnim\Graduation project model\tasnim\evaluation\statistical_analysis
Output directory: E:\tasnim\Graduation project model\tasnim\evaluation\statistical_analysis\reports
Found evaluation subdirectories: ['batches_comparison', 'detailed_batches', 'global_analysis', 'statistical_analysis']


In [3]:
def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8')
    except Exception:
        return ''

batch_comparison_reports = {}
for batch_dir in sorted((EVAL_DIR / 'batches_comparison').glob('batch_*')):
    report_path = batch_dir / 'batch_report.txt'
    batch_comparison_reports[batch_dir.name] = read_text_file(report_path)

print(f'Read {len(batch_comparison_reports)} batch comparison reports')

# Read global analysis files
global_report_text = read_text_file(EVAL_DIR / 'global_analysis' / 'GLOBAL_REPORT.txt')
global_stats = {}
global_stats_path = EVAL_DIR / 'global_analysis' / 'GLOBAL_STATS.json'
if global_stats_path.exists():
    with global_stats_path.open('r', encoding='utf-8') as f:
        global_stats = json.load(f)

print('Global analysis text length:', len(global_report_text))
print('Global stats keys:', list(global_stats.keys()))

# Read detailed batch run summaries
run_summaries = []
for batch_dir in sorted((EVAL_DIR / 'detailed_batches').glob('batch_*')):
    for run_dir in sorted(batch_dir.glob('run_*')):
        summary_path = run_dir / 'run_summary.json'
        if summary_path.exists():
            with summary_path.open('r', encoding='utf-8') as f:
                summary = json.load(f)
            run_summaries.append(summary)
print(f'Read {len(run_summaries)} detailed run summaries')


Read 5 batch comparison reports
Global analysis text length: 3173
Global stats keys: ['total_runs', 'total_batches', 'timestamp', 'batch_statistics', 'overall']
Read 25 detailed run summaries


In [4]:
# Load existing statistical analysis summary and plots
stat_summary = {}
for filename in ['global_metrics.json', 'per_batch_metrics.json', 'confidence_intervals.json', 'correlation_matrix.json', 'best_worst_runs.json']:
    path = STAT_ANALYSIS_DIR / 'summary' / filename
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            stat_summary[filename] = json.load(f)
    else:
        stat_summary[filename] = {}
        print(f'Missing summary file: {path}')

stat_report_text = read_text_file(STAT_ANALYSIS_DIR / 'reports' / 'report.txt')
stat_report_json = {}
json_path = STAT_ANALYSIS_DIR / 'reports' / 'report.json'
if json_path.exists():
    with json_path.open('r', encoding='utf-8') as f:
        stat_report_json = json.load(f)

print('Statistical analysis summary loaded:', list(stat_summary.keys()))
print('Statistical report text length:', len(stat_report_text))
print('Statistical report json keys:', list(stat_report_json.keys()))


Missing summary file: evaluation\statistical_analysis\summary\correlation_matrix.json
Statistical analysis summary loaded: ['global_metrics.json', 'per_batch_metrics.json', 'confidence_intervals.json', 'correlation_matrix.json', 'best_worst_runs.json']
Statistical report text length: 0
Statistical report json keys: ['global_metrics', 'confidence_intervals', 'per_batch_metrics', 'best_worst_runs']


In [5]:
# Build report-ready summaries
summary = {
    'runs': len(run_summaries),
    'batches': len(list((EVAL_DIR / 'detailed_batches').glob('batch_*'))),
    'batch_comparison_reports': batch_comparison_reports,
    'global_analysis_text': global_report_text,
    'global_analysis_stats': global_stats,
    'stat_analysis': stat_summary,
    'stat_report_text': stat_report_text,
    'stat_report_json': stat_report_json,
}

# Flatten run-level performance for overview
run_rows = []
for summary_item in run_summaries:
    metrics = summary_item.get('final_metrics', {})
    run_rows.append({
        'batch': summary_item.get('batch'),
        'run_name': summary_item.get('run_name'),
        'accuracy': metrics.get('accuracy'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'f1_score': metrics.get('f1'),
        'auc': metrics.get('auc'),
        'loss': metrics.get('loss')
    })

run_df = pd.DataFrame(run_rows)
print('Detailed run DataFrame shape:', run_df.shape)
print(run_df.head().to_dict(orient='records')[:2])

# Locate available plot images
plot_images = {
    'statistical_analysis': [],
    'global_analysis': [],
    'batch_comparison': []
}

for plot_dir in (STAT_ANALYSIS_DIR / 'plots').iterdir():
    if plot_dir.is_dir():
        for img in sorted(plot_dir.glob('*.png')):
            plot_images['statistical_analysis'].append(img)

for img in sorted((EVAL_DIR / 'global_analysis' / 'GLOBAL_PLOTS').glob('*.png')) if (EVAL_DIR / 'global_analysis' / 'GLOBAL_PLOTS').exists() else []:
    plot_images['global_analysis'].append(img)

for batch_dir in sorted((EVAL_DIR / 'batches_comparison').glob('batch_*')):
    comp_dir = batch_dir / 'comparison_plots'
    if comp_dir.exists():
        for img in sorted(comp_dir.glob('*.png')):
            plot_images['batch_comparison'].append(img)

print('Found plot images:')
for section, images in plot_images.items():
    print(f'  {section}: {len(images)} images')


NameError: name 'pd' is not defined

In [ ]:
def add_heading(doc: Document, text: str, level: int = 1):
    doc.add_heading(text, level=level)


def add_paragraph(doc: Document, text: str, bold: bool = False):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold
    return p


def add_table_from_dict(doc: Document, rows: List[Dict[str, Any]], columns: List[str], header_names: List[str] = None):
    table = doc.add_table(rows=1, cols=len(columns))
    table.style = 'Light Grid Accent 1'
    hdr = table.rows[0].cells
    for i, col in enumerate(columns):
        hdr[i].text = header_names[i] if header_names else col.replace('_', ' ').title()
    for row_data in rows:
        row = table.add_row().cells
        for i, col in enumerate(columns):
            value = row_data.get(col, '')
            row[i].text = f'{value}' if value is not None else ''
    return table


def add_image_if_exists(doc: Document, image_path: Path, width_inches: float = 6.0):
    if image_path.exists():
        doc.add_picture(str(image_path), width=Inches(width_inches))
        doc.add_paragraph()


def save_document(doc: Document, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(str(path))
    print(f'Saved report: {path}')


def create_full_evaluation_report(output_path: Path):
    doc = Document()
    doc.add_paragraph()
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.add_run('SecureSight Evaluation Report').bold = True
    title.runs[0].font.size = Pt(28)
    doc.add_paragraph()
    doc.add_paragraph(f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    doc.add_paragraph(f'Total runs: {summary["runs"]}')
    doc.add_paragraph(f'Total batches: {summary["batches"]}')
    doc.add_page_break()

    add_heading(doc, '1. Overview', level=1)
    add_paragraph(doc, 'This full evaluation report includes all available evaluation results from the SecureSight evaluation folder, including batch comparisons, global analysis, detailed run summaries, and current statistical analysis.')
    add_paragraph(doc, f'Number of detailed run summaries: {summary["runs"]}')
    add_paragraph(doc, f'Number of batch comparison reports: {len(summary["batch_comparison_reports"])}')

    add_heading(doc, '2. Batch Comparison Summary', level=1)
    for batch_name, report_text in summary['batch_comparison_reports'].items():
        add_heading(doc, f'{batch_name.replace("_", " ").title()}', level=2)
        add_paragraph(doc, report_text[:4000])
        add_paragraph(doc, '...')
    if plot_images['batch_comparison']:
        add_heading(doc, 'Batch Comparison Plots', level=2)
        for img in plot_images['batch_comparison'][:6]:
            add_image_if_exists(doc, img, width_inches=6)

    add_heading(doc, '3. Global Analysis', level=1)
    add_paragraph(doc, 'Global analysis results are included below from the evaluation/global_analysis folder.')
    if summary['global_analysis_text']:
        add_paragraph(doc, summary['global_analysis_text'][:5000])
    if summary['global_analysis_stats']:
        add_paragraph(doc, 'Global analysis statistics:')
        for key, value in summary['global_analysis_stats'].items():
            add_paragraph(doc, json.dumps({key: value}, indent=2))
    if plot_images['global_analysis']:
        add_heading(doc, 'Global Analysis Plots', level=2)
        for img in plot_images['global_analysis'][:6]:
            add_image_if_exists(doc, img, width_inches=6)

    add_heading(doc, '4. Detailed Batch Results Overview', level=1)
    if not run_df.empty:
        top_runs = run_df.sort_values(by='f1_score', ascending=False).head(5)
        top_rows = top_runs.to_dict(orient='records')
        add_paragraph(doc, 'Top 5 runs by F1 score:')
        add_table_from_dict(doc, top_rows, ['batch', 'run_name', 'accuracy', 'precision', 'recall', 'f1_score', 'auc', 'loss'])
    add_paragraph(doc, 'Detailed run metrics are captured from individual run summaries in the evaluation folder.')

    add_heading(doc, '5. Statistical Analysis Summary', level=1)
    add_paragraph(doc, 'This section embeds the statistical analysis generated for the evaluation results.')
    stat = summary['stat_analysis']
    if stat.get('global_metrics.json'):
        add_paragraph(doc, 'Global metrics from statistical analysis:')
        if isinstance(stat['global_metrics.json'], dict):
            for metric, metrics in stat['global_metrics.json'].items():
                add_paragraph(doc, f'{metric}: mean={metrics.get("mean")}, std={metrics.get("std")}, min={metrics.get("min")}, max={metrics.get("max")}')
    if stat.get('confidence_intervals.json'):
        add_paragraph(doc, 'Confidence intervals:')
        for metric, ci in stat['confidence_intervals.json'].items():
            add_paragraph(doc, f'{metric}: {ci.get("ci_lower")} - {ci.get("ci_upper")}')
    if stat.get('per_batch_metrics.json'):
        add_paragraph(doc, 'Per-batch metrics available for each batch.')
    if stat.get('correlation_matrix.json'):
        add_paragraph(doc, 'Correlation matrix is available in the statistical analysis outputs.')
    if plot_images['statistical_analysis']:
        add_heading(doc, 'Statistical Analysis Plots', level=2)
        for img in plot_images['statistical_analysis'][:12]:
            add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '6. Final Insights', level=1)
    add_paragraph(doc, 'This report presents a complete evaluation package for SecureSight, linking batch comparison findings with statistical analysis, global performance data, and run-level metrics.')

    save_document(doc, output_path)


def create_statistical_analysis_only_report(output_path: Path):
    doc = Document()
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.add_run('SecureSight Statistical Analysis Report').bold = True
    title.runs[0].font.size = Pt(28)
    doc.add_paragraph()
    doc.add_paragraph(f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    doc.add_paragraph(f'Total runs: {summary["runs"]}')
    doc.add_paragraph(f'Total batches: {summary["batches"]}')
    doc.add_page_break()

    add_heading(doc, '1. Overview', level=1)
    add_paragraph(doc, 'This report focuses only on the statistical analysis outputs from the SecureSight evaluation pipeline.')

    add_heading(doc, '2. Global Statistical Metrics', level=1)
    global_metrics = summary['stat_analysis'].get('global_metrics.json', {})
    if global_metrics:
        rows = []
        for metric, metrics in global_metrics.items():
            row = {'metric': metric}
            row.update({k: metrics.get(k) for k in ['mean', 'std', 'variance', 'min', 'max', 'median', 'count'] if k in metrics})
            rows.append(row)
        columns = ['metric', 'mean', 'std', 'variance', 'min', 'max', 'median', 'count']
        add_table_from_dict(doc, rows, columns, ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median', 'Count'])

    add_heading(doc, '3. Confidence Intervals', level=1)
    ci_data = summary['stat_analysis'].get('confidence_intervals.json', {})
    if ci_data:
        rows = []
        for metric, ci in ci_data.items():
            rows.append({'metric': metric, 'mean': ci.get('mean'), 'ci_lower': ci.get('ci_lower'), 'ci_upper': ci.get('ci_upper')})
        add_table_from_dict(doc, rows, ['metric', 'mean', 'ci_lower', 'ci_upper'], ['Metric', 'Mean', 'CI Lower', 'CI Upper'])

    add_heading(doc, '4. Per-Batch Statistical Metrics', level=1)
    per_batch = summary['stat_analysis'].get('per_batch_metrics.json', {})
    for batch_id, metrics in per_batch.items():
        add_heading(doc, f'Batch {batch_id}', level=2)
        rows = []
        for metric, values in metrics.items():
            row = {'metric': metric}
            row.update(values)
            rows.append(row)
        add_table_from_dict(doc, rows, ['metric', 'mean', 'std', 'variance', 'min', 'max', 'median'], ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median'])

    add_heading(doc, '5. Correlation Analysis', level=1)
    corr = summary['stat_analysis'].get('correlation_matrix.json', {})
    if corr:
        metrics = list(corr.keys())
        table = doc.add_table(rows=1 + len(metrics), cols=1 + len(metrics))
        table.style = 'Light Grid Accent 1'
        table.rows[0].cells[0].text = 'Metric'
        for i, metric in enumerate(metrics):
            table.rows[0].cells[i + 1].text = metric.upper()
        for row_i, metric1 in enumerate(metrics):
            table.rows[row_i + 1].cells[0].text = metric1.upper()
            for col_i, metric2 in enumerate(metrics):
                table.rows[row_i + 1].cells[col_i + 1].text = f'{corr[metric1].get(metric2, 0):.4f}'
    add_image_if_exists(doc, STAT_ANALYSIS_DIR / 'plots' / 'heatmaps' / 'correlation_heatmap.png', width_inches=6)

    add_heading(doc, '6. Visualizations', level=1)
    for img in plot_images['statistical_analysis'][:12]:
        add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '7. Final Insights', level=1)
    if stat_report_text:
        add_paragraph(doc, stat_report_text[:4000])

    save_document(doc, output_path)


In [ ]:
FULL_REPORT_PATH = OUTPUT_DIR / 'evaluation_full_report.docx'
STAT_ONLY_REPORT_PATH = OUTPUT_DIR / 'statistical_analysis_only_report.docx'

create_full_evaluation_report(FULL_REPORT_PATH)
create_statistical_analysis_only_report(STAT_ONLY_REPORT_PATH)

print('\nGenerated files:')
print(FULL_REPORT_PATH)
print(STAT_ONLY_REPORT_PATH)

# Validate file creation
for path in [FULL_REPORT_PATH, STAT_ONLY_REPORT_PATH]:
    if path.exists():
        print(f'Validated: {path.name} exists ({path.stat().st_size} bytes)')
    else:
        print(f'Failed: {path.name} was not created')


In [7]:
import json
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

EVAL_DIR = Path('evaluation')
STAT_ANALYSIS_DIR = EVAL_DIR / 'statistical_analysis'
OUTPUT_DIR = STAT_ANALYSIS_DIR / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper functions

def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8')
    except Exception:
        return ''


def add_heading(doc: Document, text: str, level: int = 1):
    doc.add_heading(text, level=level)


def add_paragraph(doc: Document, text: str, bold: bool = False):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.bold = bold
    return p


def add_table_from_dict(doc: Document, rows: list, columns: list, header_names: list = None):
    table = doc.add_table(rows=1, cols=len(columns))
    table.style = 'Light Grid Accent 1'
    hdr = table.rows[0].cells
    for i, col in enumerate(columns):
        hdr[i].text = header_names[i] if header_names else col.replace('_', ' ').title()
    for row_data in rows:
        row = table.add_row().cells
        for i, col in enumerate(columns):
            value = row_data.get(col, '')
            row[i].text = '' if value is None else str(value)
    return table


def add_image_if_exists(doc: Document, path: Path, width_inches: float = 6.0):
    if path.exists():
        doc.add_picture(str(path), width=Inches(width_inches))
        doc.add_paragraph()


def save_document(doc: Document, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(str(path))
    print(f'Saved report: {path}')

# Load data
batch_comparison_reports = {}
for batch_dir in sorted((EVAL_DIR / 'batches_comparison').glob('batch_*')):
    batch_comparison_reports[batch_dir.name] = read_text_file(batch_dir / 'batch_report.txt')

global_report_text = read_text_file(EVAL_DIR / 'global_analysis' / 'GLOBAL_REPORT.txt')
global_stats = {}
if (EVAL_DIR / 'global_analysis' / 'GLOBAL_STATS.json').exists():
    global_stats = json.loads((EVAL_DIR / 'global_analysis' / 'GLOBAL_STATS.json').read_text(encoding='utf-8'))

run_summaries = []
for batch_dir in sorted((EVAL_DIR / 'detailed_batches').glob('batch_*')):
    for run_dir in sorted(batch_dir.glob('run_*')):
        p = run_dir / 'run_summary.json'
        if p.exists():
            run_summaries.append(json.loads(p.read_text(encoding='utf-8')))

stat_summary = {}
for filename in ['global_metrics.json', 'per_batch_metrics.json', 'confidence_intervals.json', 'correlation_matrix.json', 'best_worst_runs.json']:
    p = STAT_ANALYSIS_DIR / 'summary' / filename
    if p.exists():
        stat_summary[filename] = json.loads(p.read_text(encoding='utf-8'))

stat_report_text = read_text_file(STAT_ANALYSIS_DIR / 'reports' / 'report.txt')
stat_report_json = {}
if (STAT_ANALYSIS_DIR / 'reports' / 'report.json').exists():
    stat_report_json = json.loads((STAT_ANALYSIS_DIR / 'reports' / 'report.json').read_text(encoding='utf-8'))

run_df = pd.DataFrame([
    {
        'batch': summary_item.get('batch'),
        'run_name': summary_item.get('run_name'),
        'accuracy': summary_item.get('final_metrics', {}).get('accuracy'),
        'precision': summary_item.get('final_metrics', {}).get('precision'),
        'recall': summary_item.get('final_metrics', {}).get('recall'),
        'f1_score': summary_item.get('final_metrics', {}).get('f1'),
        'auc': summary_item.get('final_metrics', {}).get('auc'),
        'loss': summary_item.get('final_metrics', {}).get('loss')
    }
    for summary_item in run_summaries
])

plot_images = {'statistical_analysis': [], 'global_analysis': [], 'batch_comparison': []}
for plot_dir in (STAT_ANALYSIS_DIR / 'plots').iterdir():
    if plot_dir.is_dir():
        plot_images['statistical_analysis'].extend(sorted(plot_dir.glob('*.png')))
if (EVAL_DIR / 'global_analysis' / 'GLOBAL_PLOTS').exists():
    plot_images['global_analysis'].extend(sorted((EVAL_DIR / 'global_analysis' / 'GLOBAL_PLOTS').glob('*.png')))
for batch_dir in sorted((EVAL_DIR / 'batches_comparison').glob('batch_*')):
    comp_dir = batch_dir / 'comparison_plots'
    if comp_dir.exists():
        plot_images['batch_comparison'].extend(sorted(comp_dir.glob('*.png')))

summary_data = {
    'runs': len(run_summaries),
    'batches': len(list((EVAL_DIR / 'detailed_batches').glob('batch_*'))),
    'batch_comparison_reports': batch_comparison_reports,
    'global_analysis_text': global_report_text,
    'global_analysis_stats': global_stats,
    'stat_analysis': stat_summary,
    'stat_report_text': stat_report_text,
    'stat_report_json': stat_report_json,
}

# Document generation

def create_full_evaluation_report(output_path: Path):
    doc = Document()
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.add_run('SecureSight Full Evaluation Report').bold = True
    title.runs[0].font.size = Pt(28)
    doc.add_paragraph()
    add_paragraph(doc, f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    add_paragraph(doc, f'Total runs: {summary_data["runs"]}')
    add_paragraph(doc, f'Total batches: {summary_data["batches"]}')
    doc.add_page_break()

    add_heading(doc, '1. Overview', level=1)
    add_paragraph(doc, 'This full evaluation report includes all results found in the evaluation folder, including statistical analysis, global analysis, batch comparisons, and detailed run summaries.')

    add_heading(doc, '2. Batch Comparison Summaries', level=1)
    for batch_name, report_text in summary_data['batch_comparison_reports'].items():
        add_heading(doc, batch_name.replace('_', ' ').title(), level=2)
        add_paragraph(doc, report_text[:4000])
    if plot_images['batch_comparison']:
        add_heading(doc, 'Batch Comparison Plots', level=2)
        for img in plot_images['batch_comparison'][:6]:
            add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '3. Global Analysis', level=1)
    add_paragraph(doc, 'Global analysis text and statistics from the evaluation/global_analysis folder.')
    if summary_data['global_analysis_text']:
        add_paragraph(doc, summary_data['global_analysis_text'][:5000])
    if summary_data['global_analysis_stats']:
        add_paragraph(doc, json.dumps(summary_data['global_analysis_stats'], indent=2)[:8000])
    if plot_images['global_analysis']:
        add_heading(doc, 'Global Analysis Plots', level=2)
        for img in plot_images['global_analysis'][:6]:
            add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '4. Detailed Run Summary Overview', level=1)
    if not run_df.empty:
        top_runs = run_df.sort_values(by='f1_score', ascending=False).head(5)
        add_paragraph(doc, 'Top 5 runs by F1 score:')
        add_table_from_dict(doc, top_runs.to_dict(orient='records'), ['batch', 'run_name', 'accuracy', 'precision', 'recall', 'f1_score', 'auc', 'loss'], ['Batch', 'Run Name', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC', 'Loss'])

    add_heading(doc, '5. Statistical Analysis Summary', level=1)
    if summary_data['stat_analysis'].get('global_metrics.json'):
        add_paragraph(doc, 'Global metrics:')
        for metric, metrics in summary_data['stat_analysis']['global_metrics.json'].items():
            add_paragraph(doc, f"{metric}: mean={metrics.get('mean')}, std={metrics.get('std')}, min={metrics.get('min')}, max={metrics.get('max')}")
    if summary_data['stat_analysis'].get('confidence_intervals.json'):
        add_paragraph(doc, 'Confidence intervals:')
        for metric, ci in summary_data['stat_analysis']['confidence_intervals.json'].items():
            add_paragraph(doc, f"{metric}: {ci.get('ci_lower')} - {ci.get('ci_upper')}")
    if plot_images['statistical_analysis']:
        add_heading(doc, 'Statistical Analysis Plots', level=2)
        for img in plot_images['statistical_analysis'][:12]:
            add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '6. Final Insights', level=1)
    add_paragraph(doc, 'This report captures the full evaluation workflow in a single document, combining batch comparisons, global analysis, detailed run metrics, and statistical summaries.')
    save_document(doc, output_path)


def create_statistical_analysis_only_report(output_path: Path):
    doc = Document()
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title.add_run('SecureSight Statistical Analysis Only Report').bold = True
    title.runs[0].font.size = Pt(28)
    doc.add_paragraph()
    add_paragraph(doc, f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    add_paragraph(doc, f'Total runs: {summary_data["runs"]}')
    add_paragraph(doc, f'Total batches: {summary_data["batches"]}')
    doc.add_page_break()

    add_heading(doc, '1. Overview', level=1)
    add_paragraph(doc, 'This report contains only the statistical analysis results produced for the evaluation data.')

    add_heading(doc, '2. Global Metrics', level=1)
    if summary_data['stat_analysis'].get('global_metrics.json'):
        rows = []
        for metric, metrics in summary_data['stat_analysis']['global_metrics.json'].items():
            row = {'metric': metric}
            row.update({k: metrics.get(k) for k in ['mean', 'std', 'variance', 'min', 'max', 'median', 'count'] if k in metrics})
            rows.append(row)
        add_table_from_dict(doc, rows, ['metric', 'mean', 'std', 'variance', 'min', 'max', 'median', 'count'], ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median', 'Count'])

    add_heading(doc, '3. Confidence Intervals', level=1)
    ci = summary_data['stat_analysis'].get('confidence_intervals.json', {})
    rows = [{'metric': m, 'mean': v.get('mean'), 'ci_lower': v.get('ci_lower'), 'ci_upper': v.get('ci_upper')} for m, v in ci.items()]
    if rows:
        add_table_from_dict(doc, rows, ['metric', 'mean', 'ci_lower', 'ci_upper'], ['Metric', 'Mean', 'CI Lower', 'CI Upper'])

    add_heading(doc, '4. Per-Batch Metrics', level=1)
    per_batch = summary_data['stat_analysis'].get('per_batch_metrics.json', {})
    for batch_id, metrics in per_batch.items():
        add_heading(doc, f'Batch {batch_id}', level=2)
        rows = []
        for metric, values in metrics.items():
            row = {'metric': metric}
            row.update(values)
            rows.append(row)
        add_table_from_dict(doc, rows, ['metric', 'mean', 'std', 'variance', 'min', 'max', 'median'], ['Metric', 'Mean', 'Std Dev', 'Variance', 'Min', 'Max', 'Median'])

    add_heading(doc, '5. Correlation Analysis', level=1)
    corr = summary_data['stat_analysis'].get('correlation_matrix.json', {})
    if corr:
        metrics = list(corr.keys())
        table = doc.add_table(rows=1 + len(metrics), cols=1 + len(metrics))
        table.style = 'Light Grid Accent 1'
        table.rows[0].cells[0].text = 'Metric'
        for i, metric in enumerate(metrics):
            table.rows[0].cells[i + 1].text = metric.upper()
        for row_i, metric1 in enumerate(metrics):
            table.rows[row_i + 1].cells[0].text = metric1.upper()
            for col_i, metric2 in enumerate(metrics):
                table.rows[row_i + 1].cells[col_i + 1].text = f"{corr[metric1].get(metric2, 0):.4f}"
    add_image_if_exists(doc, STAT_ANALYSIS_DIR / 'plots' / 'heatmaps' / 'correlation_heatmap.png', width_inches=6)

    add_heading(doc, '6. Visualizations', level=1)
    for img in plot_images['statistical_analysis'][:12]:
        add_image_if_exists(doc, img, width_inches=5.5)

    add_heading(doc, '7. Insights', level=1)
    if summary_data['stat_report_text']:
        add_paragraph(doc, summary_data['stat_report_text'][:4000])
    save_document(doc, output_path)

FULL_REPORT_PATH = OUTPUT_DIR / 'evaluation_full_report.docx'
STAT_ONLY_REPORT_PATH = OUTPUT_DIR / 'statistical_analysis_only_report.docx'
create_full_evaluation_report(FULL_REPORT_PATH)
create_statistical_analysis_only_report(STAT_ONLY_REPORT_PATH)
print('Generated files:')
print(FULL_REPORT_PATH)
print(STAT_ONLY_REPORT_PATH)
for path in [FULL_REPORT_PATH, STAT_ONLY_REPORT_PATH]:
    print(f'{path.name}:', 'exists' if path.exists() else 'missing', f'({path.stat().st_size} bytes)' if path.exists() else '')


Saved report: evaluation\statistical_analysis\reports\evaluation_full_report.docx
Saved report: evaluation\statistical_analysis\reports\statistical_analysis_only_report.docx
Generated files:
evaluation\statistical_analysis\reports\evaluation_full_report.docx
evaluation\statistical_analysis\reports\statistical_analysis_only_report.docx
evaluation_full_report.docx: exists (1392525 bytes)
statistical_analysis_only_report.docx: exists (210827 bytes)
